# Station-MAE — Test Results on Swiss Map

Visualises per-station MAE / RMSE on the Swiss topographic map for:
- **mr0.00** — all stations visible (pure temporal forecasting)
- **mr0.50** — 50 % stations masked (gap-filling + forecasting)

Also shows performance **grouped by 5 terrain / climate zones** and an example of the random encoder masking pattern.

---
### Terrain classification — assumptions

| Category | Altitude | Geographic filter | Climate character |
|---|---|---|---|
| Swiss Plateau / Lowland | < 600 m | LV95 northing ≥ 1 130 000 m | Temperate continental, Rhine/Rhône lowlands |
| Pre-Alpine / Jura | 600–1 400 m | LV95 northing ≥ 1 130 000 m | Humid, orographic precipitation, Jura ridges |
| Alpine | 1 400–2 200 m | LV95 northing ≥ 1 130 000 m | Mountain snow regime, sheltered valleys |
| High Alpine | > 2 200 m | any | Exposed summit regime, year-round snow |
| Southern Switzerland | < 2 000 m | LV95 northing < 1 130 000 m | Mediterranean-influenced, Ticino/Valais-Sud |

**Key assumption**: the main Alpine divide sits at approximately LV95 northing ≈ 1 130 000 m (EPSG:2056).  
Stations south of this with altitude < 2 000 m get *Southern Switzerland* regardless of elevation because their Mediterranean climate is more diagnostic than altitude.  
Summit stations (> 2 200 m) stay in *High Alpine* regardless of north/south position — their regime is altitude-driven.  
A few Valais or Graubünden stations near the watershed may be ambiguous; check the terrain map (Section 1) and adjust `ALPS_DIVIDE_NORTHING` if needed.

**Minimum sample filter**: stations with fewer than `MIN_SAMPLES` valid sensor observations for a given variable are excluded from grouped statistics but still shown on maps.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import geopandas as gpd
import rioxarray  # noqa — registers .rio accessor on xarray DataArrays

# ═══════════════════════════════════════════════════════════════════════════
#  CONFIGURE PATHS — update these for your local machine
# ═══════════════════════════════════════════════════════════════════════════
NOTEBOOK_DIR = os.path.abspath('')   # folder containing this notebook

# Root of the PeakWeatherDataset download (contains /obs/, /static/, etc.)
DATA_ROOT = '/Users/aureliedejong/Documents/ETH/_DAS Project/PeakWeatherDataset'

# swissBOUNDARIES3D shapefile (.shp.zip or unpacked .shp)
PATH_SWISSSHAPE = '/Users/aureliedejong/Documents/ETH/_DAS Project/swissboundaries3d_2026-01_2056_5728.shp.zip'

# Test results directory (relative or absolute)
RESULTS_DIR = os.path.join(NOTEBOOK_DIR, 'test_results')
# ═══════════════════════════════════════════════════════════════════════════

# ── Terrain classification parameters ──────────────────────────────────────
# LV95 northing (metres) of the main Alpine divide — stations south of this
# with altitude < 2000 m are classified as 'Southern Switzerland'.
# Adjust slightly if border stations look misclassified on the terrain map.
ALPS_DIVIDE_NORTHING = 1_130_000

# Minimum valid sensor readings per station to include in grouped stats.
# Stations below this threshold for a given variable are excluded from that
# variable's grouped statistics but still plotted on maps.
MIN_SAMPLES = 50

# ── Variable metadata ───────────────────────────────────────────────────────
# (name, csv_column_for_mae, physical_unit, colormap, (vmin, vmax))
# plasma_r: yellow (low/good) → magenta → purple (high/bad)
# No green or yellow-green → high contrast against terrain DEM background.
VARIABLES = [
    ('temperature', 'temperature_mae', 'deg C', 'plasma_r', (0, 3.5)),
    ('pressure',    'pressure_mae',    'hPa',   'plasma_r', (0, 5.0)),
    ('humidity',    'humidity_mae',    '%',     'plasma_r', (0, 20.)),
    ('wind_u',      'wind_u_mae',      'm/s',   'plasma_r', (0, 3.0)),
    ('wind_v',      'wind_v_mae',      'm/s',   'plasma_r', (0, 3.0)),
]

# ── Terrain category display config ────────────────────────────────────────
TERRAIN_CATEGORIES = [
    'Swiss Plateau / Lowland',
    'Pre-Alpine / Jura',
    'Alpine',
    'High Alpine',
    'Southern Switzerland',
]
TERRAIN_COLORS = {
    'Swiss Plateau / Lowland': '#4CAF50',
    'Pre-Alpine / Jura':       '#2196F3',
    'Alpine':                  '#9C27B0',
    'High Alpine':             '#F44336',
    'Southern Switzerland':    '#FF9800',
}
TERRAIN_SHORT = {
    'Swiss Plateau / Lowland': 'Plateau',
    'Pre-Alpine / Jura':       'Pre-Alp',
    'Alpine':                  'Alpine',
    'High Alpine':             'High Alp',
    'Southern Switzerland':    'South CH',
}

if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

print('NOTEBOOK_DIR :', NOTEBOOK_DIR)
print('RESULTS_DIR  :', RESULTS_DIR)
print('Contents     :', sorted(os.listdir(RESULTS_DIR)))

In [ ]:
# ── Load PeakWeatherDataset (station metadata + DEM) ─────────────────────────
# We load with extended_topo_vars='DEM' so that ds.load_topography()
# returns the elevation raster needed for map backgrounds.
# Using freq='d' (daily) instead of '10min' to skip loading the full
# time series — we only need static station metadata and the DEM here.
from peakweather.dataset import PeakWeatherDataset

ds = PeakWeatherDataset(
    root=DATA_ROOT,
    parameters=['temperature', 'pressure', 'humidity',
                'wind_speed', 'wind_direction', 'precipitation'],
    compute_uv=True,
    station_type='meteo_station',
    imputation_method=None,
    freq='d',                    # daily — fast to load; we only need static metadata
    extended_topo_vars='DEM',    # required for ds.load_topography()['topo_DEM']
)
print(f'Loaded {len(ds.stations_table)} stations')
print('stations_table columns:', list(ds.stations_table.columns))
ds.stations_table.head(3)

In [ ]:
# ── Build station_idx → LV95 coordinate mapping ───────────────────────────────
# The training code calls dataset.exclude_stations(['PFA']), then enumerates
# the remaining rows of stations_table as station_idx 0, 1, 2, ...
# We replicate that ordering so station_idx matches per_station_metrics.csv.
#
# nat_abbr is the DataFrame index — reset_index() promotes it to a column.

EXCLUDE = ['PFA']
stns = ds.stations_table.copy()
excl_upper = {s.upper() for s in EXCLUDE}

keep_mask = []
for idx, row in stns.iterrows():
    candidates = {str(idx).upper()}
    keep_mask.append(not bool(candidates & excl_upper))

stns_filtered = stns[keep_mask].reset_index(drop=False)   # nat_abbr → column
stns_filtered['station_idx'] = range(len(stns_filtered))

print(f'Stations after excluding {EXCLUDE}: {len(stns_filtered)}')

# The columns we need from stations_table:
#   swiss_easting / swiss_northing — raw LV95 coordinates (metres)
#   station_height                 — elevation above sea level (m)
#   nat_abbr                       — MeteoSwiss station abbreviation (for labels)
coord_cols = ['station_idx', 'swiss_easting', 'swiss_northing', 'station_height']
if 'nat_abbr' in stns_filtered.columns:
    coord_cols.append('nat_abbr')

coords = stns_filtered[coord_cols].rename(
    columns={'swiss_easting': 'e_lv95', 'swiss_northing': 'n_lv95'}
)
print(coords.head(5).to_string())

In [ ]:
# ── Load per-station metrics and merge with LV95 coordinates ──────────────────
df_mr0 = pd.read_csv(os.path.join(RESULTS_DIR, 'best_mr0.00', 'per_station_metrics.csv'))
df_mr5 = pd.read_csv(os.path.join(RESULTS_DIR, 'best_mr0.50', 'per_station_metrics.csv'))

df_mr0 = df_mr0.merge(coords, on='station_idx', how='left')
df_mr5 = df_mr5.merge(coords, on='station_idx', how='left')
print(f'mr0.00: {len(df_mr0)} stations   mr0.50: {len(df_mr5)} stations')

# ── Terrain classification ────────────────────────────────────────────────────
def classify_terrain(df):
    """
    Classify each station into one of 5 terrain/climate categories.

    Priority order (first match wins):
      1. High Alpine    : station_height > 2200 m  (any position)
      2. Southern CH    : n_lv95 < ALPS_DIVIDE_NORTHING  AND  height < 2000 m
      3. Plateau        : height < 600 m
      4. Pre-Alpine     : 600 <= height < 1400 m
      5. Alpine         : 1400 <= height <= 2200 m
    """
    cats = []
    for _, row in df.iterrows():
        h = row.get('station_height', np.nan)
        n = row.get('n_lv95',         np.nan)
        if pd.isna(h) or pd.isna(n):
            cats.append('Unknown')
            continue
        if h > 2200:
            cats.append('High Alpine')
        elif n < ALPS_DIVIDE_NORTHING and h < 2000:
            cats.append('Southern Switzerland')
        elif h < 600:
            cats.append('Swiss Plateau / Lowland')
        elif h < 1400:
            cats.append('Pre-Alpine / Jura')
        else:
            cats.append('Alpine')
    return cats

df_mr0['terrain'] = classify_terrain(df_mr0)
df_mr5['terrain'] = classify_terrain(df_mr5)

print('\nStation counts per terrain category:')
counts = df_mr0['terrain'].value_counts()
for cat in TERRAIN_CATEGORIES:
    n = counts.get(cat, 0)
    print(f'  {cat:<28}  {n:>3} stations')
if counts.get('Unknown', 0) > 0:
    print(f'  {"Unknown":<28}  {counts["Unknown"]:>3}  <- check coordinates!')

df_mr0[['station_idx','e_lv95','n_lv95','station_height','terrain','temperature_mae']].head(6)

In [ ]:
# ── Load raw predictions ──────────────────────────────────────────────────────
pred_mr5 = torch.load(
    os.path.join(RESULTS_DIR, 'best_mr0.50', 'predictions.pt'),
    map_location='cpu', weights_only=False
)
pred_mr0 = torch.load(
    os.path.join(RESULTS_DIR, 'best_mr0.00', 'predictions.pt'),
    map_location='cpu', weights_only=False
)
print('mr0.50 tensor shapes:')
for k, v in pred_mr5.items():
    print(f'  {k:<14}: {tuple(v.shape) if hasattr(v, "shape") else v}')

In [ ]:
# ── Swiss map helpers ─────────────────────────────────────────────────────────

def _load_dem_and_border(ds, path_swissshape, coarsen=10):
    """Load DEM raster + CH border. Call once and reuse the result."""
    switzerland = gpd.read_file(
        path_swissshape,
        layer='swissBOUNDARIES3D_1_5_TLM_LANDESGEBIET',
    ).to_crs('EPSG:2056')
    minx, miny, maxx, maxy = switzerland.total_bounds

    topo   = ds.load_topography()
    dem    = topo['topo_DEM'].dem
    dem_ch = dem.rio.clip(switzerland.geometry, switzerland.crs, drop=False)

    dem_bg = dem.coarsen(x=coarsen, y=coarsen, boundary='trim').mean()
    dem_fg = dem_ch.coarsen(x=coarsen, y=coarsen, boundary='trim').mean()

    dem_bg = dem_bg.sel(x=slice(minx, maxx), y=slice(miny, maxy))
    dem_fg = dem_fg.sel(x=slice(minx, maxx), y=slice(miny, maxy))
    return dem_bg, dem_fg, switzerland


def draw_dem(ax, dem_bg, dem_fg, switzerland):
    """Render DEM background + Swiss border onto ax."""
    norm = mcolors.Normalize(vmin=0, vmax=4500)
    dem_bg.plot(ax=ax, cmap='terrain', norm=norm, alpha=0.35,
                robust=True, add_labels=False, add_colorbar=False)
    dem_fg.plot(ax=ax, cmap='terrain', norm=norm,
                robust=True, add_labels=False, add_colorbar=False)
    switzerland.boundary.plot(ax=ax, color='white', linewidth=1.0)
    ax.axis('off')


def scatter_metric(ax, df, metric_col, cmap='plasma_r',
                   vmin=None, vmax=None, size=60,
                   edgecolor='white', linewidth=0.4, zorder=5):
    """Scatter stations on ax, coloured by a continuous metric value."""
    valid = df.dropna(subset=['e_lv95', 'n_lv95', metric_col])
    return ax.scatter(
        valid['e_lv95'], valid['n_lv95'],
        c=valid[metric_col], cmap=cmap, vmin=vmin, vmax=vmax,
        s=size, edgecolors=edgecolor, linewidths=linewidth, zorder=zorder,
    )


print('Loading DEM + border (first call takes ~30 s) ...')
dem_bg, dem_fg, switzerland = _load_dem_and_border(ds, PATH_SWISSSHAPE)
print('Done.')

---
## 1 — Terrain category map

Verify the classification looks geographically sensible before interpreting grouped performance.  
The dashed line shows the Alpine-divide northing threshold used to separate Southern Switzerland.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))
draw_dem(ax, dem_bg, dem_fg, switzerland)

for cat in TERRAIN_CATEGORIES:
    sub = df_mr0[df_mr0['terrain'] == cat].dropna(subset=['e_lv95', 'n_lv95'])
    ax.scatter(
        sub['e_lv95'], sub['n_lv95'],
        color=TERRAIN_COLORS[cat], s=65,
        edgecolors='white', linewidths=0.5, zorder=8,
        label=f'{cat}  (n={len(sub)})',
    )

ax.axhline(ALPS_DIVIDE_NORTHING, color='navy', linewidth=0.9, linestyle='--', alpha=0.6,
           label=f'Alps divide threshold ({ALPS_DIVIDE_NORTHING/1e6:.3f} M m northing)')

ax.legend(fontsize=9, framealpha=0.9, loc='upper left',
          title='Terrain category', title_fontsize=9)
ax.set_title('Station terrain / climate classification', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'map_terrain_categories.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 2 — Per-variable MAE maps: mr0.00 (all stations visible)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(28, 5))
fig.suptitle('MAE by station — mr0.00 (all stations visible)', fontsize=14, y=1.01)

for ax, (var, col, unit, cmap, (lo, hi)) in zip(axes, VARIABLES):
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    sc = scatter_metric(ax, df_mr0, col, cmap=cmap, vmin=lo, vmax=hi, size=55)
    plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label=unit)
    ax.set_title(var, fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'map_mae_mr0.00.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 3 — Per-variable MAE maps: mr0.50 (50 % stations masked)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(28, 5))
fig.suptitle('MAE by station — mr0.50 (gap-filling + forecasting)', fontsize=14, y=1.01)

for ax, (var, col, unit, cmap, (lo, hi)) in zip(axes, VARIABLES):
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    sc = scatter_metric(ax, df_mr5, col, cmap=cmap, vmin=lo, vmax=hi, size=55)
    plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label=unit)
    ax.set_title(var, fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'map_mae_mr0.50.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 4 — Overall RMSE map (both mask ratios, side by side)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, (df, title) in zip(axes, [
    (df_mr0, 'Overall RMSE — mr0.00 (no masking)'),
    (df_mr5, 'Overall RMSE — mr0.50 (50% masked)'),
]):
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    sc = scatter_metric(ax, df, 'overall_rmse_norm', cmap='plasma_r', vmin=0.3, vmax=1.0, size=60)
    plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label='RMSE (norm.)')
    ax.set_title(title, fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'map_overall_rmse.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 5 — Masking penalty map: Δ MAE = mr0.50 − mr0.00

**Red** = stations that are harder to predict when masked from the encoder.  
**White / green** = stations the model reconstructs well even without seeing their own input.

In [ ]:
diff = df_mr0[['station_idx', 'e_lv95', 'n_lv95', 'station_height', 'terrain']].copy()
for var, col, unit, _, _ in VARIABLES:
    diff[f'd_{col}'] = df_mr5[col].values - df_mr0[col].values
diff['d_overall_mae'] = df_mr5['overall_mae_norm'].values - df_mr0['overall_mae_norm'].values

fig, axes = plt.subplots(1, 5, figsize=(28, 5))
fig.suptitle('Masking penalty DMAE = mr0.50 - mr0.00  (red = harder when masked)',
             fontsize=13, y=1.01)
for ax, (var, col, unit, _, _) in zip(axes, VARIABLES):
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    sc = scatter_metric(ax, diff, f'd_{col}', cmap='RdBu_r', vmin=-0.5, vmax=0.5, size=55)
    plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label=unit)
    ax.set_title(var, fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'map_masking_penalty.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 6 — Terrain grouped statistics

Mean ± std of per-station MAE for each terrain category and variable.  
Stations with < `MIN_SAMPLES` valid observations for a variable are excluded from that cell.

In [ ]:
def terrain_grouped_stats(df):
    records = []
    for cat in TERRAIN_CATEGORIES:
        sub = df[df['terrain'] == cat]
        row = {'terrain': cat, 'n_stations': len(sub)}
        for var, col, unit, _, _ in VARIABLES:
            n_col = f'{var}_n_samples'
            if n_col in sub.columns:
                valid = sub[sub[n_col] >= MIN_SAMPLES][col].dropna()
            else:
                valid = sub[col].dropna()
            row[f'{var}_mae']  = round(valid.mean(), 3) if len(valid) else float('nan')
            row[f'{var}_std']  = round(valid.std(),  3) if len(valid) else float('nan')
            row[f'{var}_n']    = len(valid)
        records.append(row)
    return pd.DataFrame(records).set_index('terrain')

stats_mr0 = terrain_grouped_stats(df_mr0)
stats_mr5 = terrain_grouped_stats(df_mr5)

mean_cols = [f'{var}_mae' for var, *_ in VARIABLES]
std_cols  = [f'{var}_std' for var, *_ in VARIABLES]

print('Mean MAE by terrain — mr0.00 (no masking) [physical units]')
display(stats_mr0[['n_stations'] + mean_cols])
print('\nMean MAE by terrain — mr0.50 (50% masked) [physical units]')
display(stats_mr5[['n_stations'] + mean_cols])

In [ ]:
# ── Box plots: MAE distribution by terrain category and variable ──────────────
# Solid fill = mr0.00 (no masking)  |  Hatched = mr0.50 (50% masked)

fig, axes = plt.subplots(1, 5, figsize=(26, 5))
fig.suptitle(
    'Per-station MAE distribution by terrain category\n'
    '(solid = mr0.00 no masking  |  hatched = mr0.50 50% masked)',
    fontsize=12, y=1.04
)

cat_colors = [TERRAIN_COLORS[c] for c in TERRAIN_CATEGORIES]
x = np.arange(len(TERRAIN_CATEGORIES))
w = 0.35

for ax, (var, col, unit, _, (lo, hi)) in zip(axes, VARIABLES):
    n_col = f'{var}_n_samples'

    def get_vals(df, cat):
        sub = df[df['terrain'] == cat]
        if n_col in sub.columns:
            return sub[sub[n_col] >= MIN_SAMPLES][col].dropna().values
        return sub[col].dropna().values

    data0 = [get_vals(df_mr0, c) for c in TERRAIN_CATEGORIES]
    data5 = [get_vals(df_mr5, c) for c in TERRAIN_CATEGORIES]

    bp0 = ax.boxplot(data0, positions=x - w/2, widths=w*0.9,
                     patch_artist=True, showfliers=False,
                     medianprops=dict(color='black', linewidth=1.5))
    bp5 = ax.boxplot(data5, positions=x + w/2, widths=w*0.9,
                     patch_artist=True, showfliers=False,
                     medianprops=dict(color='black', linewidth=1.5))

    for patch, color in zip(bp0['boxes'], cat_colors):
        patch.set_facecolor(color); patch.set_alpha(0.85)
    for patch, color in zip(bp5['boxes'], cat_colors):
        patch.set_facecolor(color); patch.set_alpha(0.45); patch.set_hatch('//')

    ax.set_xticks(x)
    ax.set_xticklabels([TERRAIN_SHORT[c] for c in TERRAIN_CATEGORIES],
                       fontsize=7, rotation=15)
    ax.set_ylabel(f'MAE ({unit})', fontsize=8)
    ax.set_title(var, fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(bottom=0)

leg = [
    mpatches.Patch(facecolor='grey', alpha=0.85, label='mr0.00 (no masking)'),
    mpatches.Patch(facecolor='grey', alpha=0.45, hatch='//', label='mr0.50 (50% masked)'),
]
axes[-1].legend(handles=leg, fontsize=8, loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'boxplot_terrain_mae.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 7 — Masking penalty by terrain category

Which terrain types suffer most when stations are hidden from the encoder?  
A large positive bar = the model relies on nearby stations to interpolate that region,
so hiding them hurts. Near-zero / negative = the model generalises well from temporal context alone.

In [ ]:
VAR_COLORS = ['#E53935', '#8E24AA', '#1E88E5', '#43A047', '#FB8C00']

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(TERRAIN_CATEGORIES))
w = 0.14

for i, (var, col, unit, _, _) in enumerate(VARIABLES):
    d_col = f'd_{col}'
    means, stds = [], []
    for cat in TERRAIN_CATEGORIES:
        vals = diff[diff['terrain'] == cat][d_col].dropna()
        means.append(vals.mean())
        stds.append(vals.std())
    offset = (i - 2) * w
    ax.bar(x + offset, means, width=w*0.9, color=VAR_COLORS[i], alpha=0.85, label=var)
    ax.errorbar(x + offset, means, yerr=stds,
                fmt='none', color='black', capsize=2.5, linewidth=0.8)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels([TERRAIN_SHORT[c] for c in TERRAIN_CATEGORIES], fontsize=10)
ax.set_ylabel('DMAE = mr0.50 - mr0.00 (physical units)', fontsize=9)
ax.set_title(
    'Masking penalty by terrain  (positive = harder when masked | '
    'negative = helped by masking)',
    fontsize=10
)
ax.legend(fontsize=9, loc='upper right', ncol=5)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'terrain_masking_penalty.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 8 — Temperature MAE: dot fill = error, outline = terrain category

Combine the geographic MAE pattern with terrain classification in a single map.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

for ax, (df, title) in zip(axes, [
    (df_mr0, 'Temperature MAE — mr0.00'),
    (df_mr5, 'Temperature MAE — mr0.50'),
]):
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    sc_last = None
    for cat in TERRAIN_CATEGORIES:
        sub = df[df['terrain'] == cat].dropna(subset=['e_lv95', 'n_lv95', 'temperature_mae'])
        lo, hi = VARIABLES[0][4]
        sc_last = ax.scatter(
            sub['e_lv95'], sub['n_lv95'],
            c=sub['temperature_mae'], cmap='plasma_r', vmin=lo, vmax=hi,
            s=70, edgecolors=TERRAIN_COLORS[cat], linewidths=1.8, zorder=8,
        )
    if sc_last is not None:
        plt.colorbar(sc_last, ax=ax, fraction=0.035, pad=0.02, label='deg C')
    ax.set_title(title, fontsize=11)

# Terrain legend (outline colours)
handles = [
    mpatches.Patch(edgecolor=TERRAIN_COLORS[c], facecolor='#DDDDDD',
                   linewidth=1.8, label=c)
    for c in TERRAIN_CATEGORIES
]
axes[0].legend(handles=handles, fontsize=8, title='Terrain (outline)', loc='upper left')
plt.suptitle(
    'Temperature MAE — dot fill = error magnitude  |  outline colour = terrain category',
    fontsize=12
)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'map_temperature_mae_terrain.png'),
            dpi=150, bbox_inches='tight')
plt.show()

---
## 9 — Station altitude vs MAE scatter (coloured by terrain)

Circles = mr0.00  |  Triangles = mr0.50

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(26, 4))
fig.suptitle(
    'Station altitude vs MAE by terrain  '
    '(circle = mr0.00  |  triangle = mr0.50)',
    fontsize=12, y=1.02
)

for ax, (var, col, unit, _, _) in zip(axes, VARIABLES):
    for cat in TERRAIN_CATEGORIES:
        c = TERRAIN_COLORS[cat]
        s0 = df_mr0[df_mr0['terrain'] == cat].dropna(subset=['station_height', col])
        s5 = df_mr5[df_mr5['terrain'] == cat].dropna(subset=['station_height', col])
        ax.scatter(s0['station_height'], s0[col], color=c, alpha=0.8,
                   s=25, marker='o', label=cat)
        ax.scatter(s5['station_height'], s5[col], color=c, alpha=0.35,
                   s=25, marker='^')
    ax.set_xlabel('Altitude (m)', fontsize=8)
    ax.set_ylabel(f'MAE ({unit})', fontsize=8)
    ax.set_title(var, fontsize=10)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)

color_handles = [
    mpatches.Patch(color=TERRAIN_COLORS[c], label=c) for c in TERRAIN_CATEGORIES
]
shape_handles = [
    plt.scatter([], [], marker='o', color='grey', alpha=0.8, s=25, label='mr0.00'),
    plt.scatter([], [], marker='^', color='grey', alpha=0.35, s=25, label='mr0.50'),
]
axes[0].legend(handles=color_handles + shape_handles, fontsize=7, loc='upper left',
               title='Terrain / mask', title_fontsize=7, ncol=1)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'scatter_altitude_mae_terrain.png'),
            dpi=150, bbox_inches='tight')
plt.show()

---
## 10 — Masked stations example

Shows which stations the encoder sees vs. which it must reconstruct for one test window.

In [ ]:
WINDOW_IDX = 42    # change to inspect other windows
MASK_RATIO  = 0.50

y_mask = pred_mr5['masks']   # (M, K, N, V)
N      = y_mask.shape[2]

# Sensor availability (any variable with data at the first delta step)
has_data = y_mask[WINDOW_IDX, 0, :, :5].any(dim=-1).numpy()

# Simulate random encoder mask (same logic as model/encoder.py _mask_stations)
rng = torch.Generator()
rng.manual_seed(WINDOW_IDX)
perm       = torch.randperm(N, generator=rng)
n_masked   = int(N * MASK_RATIO)
masked_set = set(perm[:n_masked].tolist())

e_all = df_mr5['e_lv95'].values
n_all = df_mr5['n_lv95'].values

status = []
for i in range(len(df_mr5)):
    if not has_data[i]:    status.append('no_data')
    elif i in masked_set:  status.append('masked')
    else:                  status.append('visible')
status = np.array(status)

fig, ax = plt.subplots(figsize=(11, 8))
draw_dem(ax, dem_bg, dem_fg, switzerland)

for grp, color, marker, sz, lab, zo in [
    ('visible', '#2196F3', 'o', 60, 'Visible (context)',       8),
    ('masked',  '#F44336', 'X', 75, 'Masked (model predicts)', 10),
    ('no_data', '#AAAAAA', 'o', 30, 'No sensor data',          6),
]:
    sel = status == grp
    ax.scatter(e_all[sel], n_all[sel], color=color, marker=marker, s=sz,
               edgecolors='white', linewidths=0.5, label=lab, zorder=zo)

ax.legend(fontsize=9, framealpha=0.85, loc='upper left')
ax.set_title(
    f'Encoder masking — window #{WINDOW_IDX}   '
    f'({n_masked}/{N} stations hidden)',
    fontsize=11,
)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'map_masked_stations_example.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f'Visible: {(status=="visible").sum()}  '
      f'Masked: {(status=="masked").sum()}  '
      f'No data: {(status=="no_data").sum()}')

---
## 11 — Best / worst stations by temperature MAE

In [ ]:
TOP_N = 10
base_cols = ['station_idx', 'station_height', 'terrain',
             'temperature_mae', 'temperature_rmse', 'overall_mae_norm']
if 'nat_abbr' in df_mr0.columns:
    base_cols = ['nat_abbr'] + base_cols

for label, df in [('mr0.00', df_mr0), ('mr0.50', df_mr5)]:
    sub = df.dropna(subset=['temperature_mae']).sort_values('temperature_mae')
    print(f'\n-- {label} Top-{TOP_N} BEST temperature MAE')
    display(sub[base_cols].head(TOP_N).reset_index(drop=True))
    print(f'\n-- {label} Top-{TOP_N} WORST temperature MAE')
    display(sub[base_cols].tail(TOP_N).reset_index(drop=True))

---
## 12 — Humidity RMSE spatial pattern

Humidity shows an anomalously high spike at nowcast (delta=0).  
Map it to see if the issue clusters geographically.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, (df, title) in zip(axes, [
    (df_mr0, 'Humidity RMSE — mr0.00'),
    (df_mr5, 'Humidity RMSE — mr0.50'),
]):
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    sc = scatter_metric(ax, df, 'humidity_rmse', cmap='RdPu', vmin=0, vmax=30, size=60)
    plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label='%')
    ax.set_title(title, fontsize=11)
plt.suptitle('Humidity RMSE (%)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'map_humidity_rmse.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 13 — Raw prediction vs target: time series for best and worst station

Picks the station with the best and worst temperature MAE (mr0.50), 30-min lead time.

In [ ]:
VAR_IDX   = 0   # 0=temperature  1=pressure  2=humidity  3=wind_u  4=wind_v
DELTA_IDX = 1   # K index: 0=0 min  1=30 min  2=60 min ...
VAR_NAME  = 'temperature'

preds_t   = pred_mr5['preds']    # (M, K, N, V)
targets_t = pred_mr5['targets']  # (M, K, N, V)
masks_t   = pred_mr5['masks']    # (M, K, N, V)

sub = df_mr5.dropna(subset=['temperature_mae'])
best_idx  = int(sub.nsmallest(1, 'temperature_mae')['station_idx'].values[0])
worst_idx = int(sub.nlargest(1,  'temperature_mae')['station_idx'].values[0])

for idx, tag in [(best_idx, 'Best'), (worst_idx, 'Worst')]:
    row = sub[sub['station_idx'] == idx].iloc[0]
    abbr = row.get('nat_abbr', f'idx={idx}')
    print(f'{tag}: {abbr}  |  {row["station_height"]:.0f} m  |  {row["terrain"]}  '
          f'|  temperature_mae={row["temperature_mae"]:.3f} deg C')

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

for ax, (idx, title_tag) in zip(axes, [(best_idx, 'Best'), (worst_idx, 'Worst')]):
    row  = sub[sub['station_idx'] == idx].iloc[0]
    abbr = row.get('nat_abbr', f'idx={idx}')
    cat  = row.get('terrain', '?')

    pred_vals   = preds_t[:, DELTA_IDX, idx, VAR_IDX].numpy()
    target_vals = targets_t[:, DELTA_IDX, idx, VAR_IDX].numpy()
    valid       = masks_t[:, DELTA_IDX, idx, VAR_IDX].numpy().astype(bool)

    x = np.arange(len(pred_vals))
    # Use light grey for target (visible on both dark and light backgrounds)
    ax.plot(x[valid], target_vals[valid], color='#CCCCCC', lw=1.4, alpha=0.95, label='target')
    ax.plot(x[valid], pred_vals[valid],   color='#FF4444', lw=1.1, alpha=0.9,
            linestyle='--', label='pred')
    mae = np.abs(pred_vals[valid] - target_vals[valid]).mean()
    ax.set_title(
        f'{title_tag}: {abbr}  ({cat}, {row["station_height"]:.0f} m)  '
        f'temperature @30 min  MAE={mae:.3f} (norm.)',
        fontsize=10
    )
    ax.set_ylabel('Norm. value')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

axes[-1].set_xlabel('Test window index')
plt.suptitle(f'{VAR_NAME} predictions — mr0.50 — {len(x)} windows', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'timeseries_best_worst.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 14 — Persistence baseline comparison

Per-station RMSE for the model vs a persistence forecast (last observed value held constant).  
Lead times **k = 1 … 12** only (30 min → 6 h); k = 0 is excluded because persistence is trivially perfect there.

**Skill score** = 1 − model\_RMSE / persistence\_RMSE  
→ **positive (blue)** = model beats persistence  
→ **negative (red)** = persistence beats model

In [ ]:
# ── Plot: model vs persistence vs per-station skill score ─────────────────────
# 3 maps side by side, shared color scale for RMSE panels (plasma_r),
# diverging RdBu for skill (blue = model beats persistence).

VAR_NAMES_5 = ['temperature', 'pressure', 'humidity', 'wind_u', 'wind_v']

# Compute per-station RMSE from raw tensors (k > 0 only, fair comparison)
def _per_station_rmse_from_tensors(pred_dict, label=''):
    """
    Compute per-station overall RMSE from a predictions.pt dict.
    Uses lead times k=1..K-1 (excludes nowcast k=0).

    Returns a DataFrame with columns: station_idx, overall_rmse_norm,
    {var}_rmse_norm for each of the 5 target variables.
    """
    preds   = pred_dict['preds']              # (M, K, N, 5)
    targets = pred_dict['targets'][:, :, :, :5]  # (M, K, N, 5)
    masks   = pred_dict['masks'  ][:, :, :, :5]  # (M, K, N, 5)

    # k=0 target used as persistence forecast for all lead times
    persist = targets[:, 0:1, :, :]          # (M, 1, N, 5)

    # Errors for k > 0 only (30 min → 6 h)
    model_sq   = (preds   [:, 1:, :, :] - targets[:, 1:, :, :]) ** 2  # (M, K-1, N, 5)
    persist_sq = (persist.expand_as(preds[:, 1:, :, :]) - targets[:, 1:, :, :]) ** 2
    mask_fwd   = masks[:, 1:, :, :].bool()                             # (M, K-1, N, 5)

    N = preds.shape[2]
    rows_m, rows_p = [], []
    for n in range(N):
        rm = {'station_idx': n}
        rp = {'station_idx': n}
        sq_sum_m = sq_sum_p = 0.0
        n_vars = 0
        for v, var in enumerate(VAR_NAMES_5):
            m_nv = mask_fwd[:, :, n, v]
            cnt  = int(m_nv.sum().item())
            if cnt > 0:
                ms = float(model_sq  [:, :, n, v][m_nv].mean().item())
                ps = float(persist_sq[:, :, n, v][m_nv].mean().item())
                rm[f'{var}_rmse_norm'] = ms ** 0.5
                rp[f'{var}_rmse_norm'] = ps ** 0.5
                sq_sum_m += ms
                sq_sum_p += ps
                n_vars   += 1
            else:
                rm[f'{var}_rmse_norm'] = float('nan')
                rp[f'{var}_rmse_norm'] = float('nan')
        rm['overall_rmse_norm'] = (sq_sum_m / n_vars) ** 0.5 if n_vars else float('nan')
        rp['overall_rmse_norm'] = (sq_sum_p / n_vars) ** 0.5 if n_vars else float('nan')
        rows_m.append(rm)
        rows_p.append(rp)

    df_m = pd.DataFrame(rows_m).merge(coords, on='station_idx', how='left')
    df_p = pd.DataFrame(rows_p).merge(coords, on='station_idx', how='left')
    df_m['terrain'] = classify_terrain(df_m)
    df_p['terrain'] = classify_terrain(df_p)
    if label:
        print(f'{label} — overall RMSE (norm, k>0)  '
              f'model median={df_m["overall_rmse_norm"].median():.3f}  '
              f'persist median={df_p["overall_rmse_norm"].median():.3f}')
    return df_m, df_p


df_model_mr0_fwd, df_persist_mr0 = _per_station_rmse_from_tensors(pred_mr0, 'mr0.00')
df_model_mr5_fwd, df_persist_mr5 = _per_station_rmse_from_tensors(pred_mr5, 'mr0.50')

# Per-station skill score:  skill = 1 - model_rmse / persist_rmse
# Positive (blue) → model beats persistence
# Negative (red)  → persistence beats model
for df_m, df_p, tag in [
    (df_model_mr0_fwd, df_persist_mr0, 'mr0.00'),
    (df_model_mr5_fwd, df_persist_mr5, 'mr0.50'),
]:
    skill_col = df_m[['station_idx', 'e_lv95', 'n_lv95', 'nat_abbr', 'terrain']].copy()
    skill_col['skill'] = 1.0 - df_m['overall_rmse_norm'].values / df_p['overall_rmse_norm'].values
    if tag == 'mr0.00':
        df_skill_mr0 = skill_col
    else:
        df_skill_mr5 = skill_col

# Shared RMSE color scale: choose vmin/vmax that covers both model and persistence
_all_rmse = pd.concat([
    df_model_mr0_fwd['overall_rmse_norm'], df_persist_mr0['overall_rmse_norm'],
    df_model_mr5_fwd['overall_rmse_norm'], df_persist_mr5['overall_rmse_norm'],
]).dropna()
RMSE_VMIN = float(_all_rmse.quantile(0.02))
RMSE_VMAX = float(_all_rmse.quantile(0.98))
print(f'Shared RMSE color scale: [{RMSE_VMIN:.3f}, {RMSE_VMAX:.3f}]')

# ── Figure: 3 columns × 2 rows (mr0.00 top / mr0.50 bottom) ───────────────
fig, axes = plt.subplots(2, 3, figsize=(27, 12))
fig.suptitle(
    'Per-station RMSE — model vs persistence (lead times 30 min → 6 h)\n'
    'Skill = 1 − model/persistence  (blue = model better  |  red = persistence better)',
    fontsize=13, y=1.01
)

for row_idx, (df_m, df_p, df_sk, mr_tag) in enumerate([
    (df_model_mr0_fwd, df_persist_mr0, df_skill_mr0, 'mr0.00 (all visible)'),
    (df_model_mr5_fwd, df_persist_mr5, df_skill_mr5, 'mr0.50 (50% masked)'),
]):
    # Left: model RMSE
    ax = axes[row_idx, 0]
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    sc = scatter_metric(ax, df_m, 'overall_rmse_norm', cmap='plasma_r',
                        vmin=RMSE_VMIN, vmax=RMSE_VMAX, size=60)
    plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label='RMSE (norm.)')
    ax.set_title(f'Model — {mr_tag}', fontsize=11)

    # Centre: persistence RMSE (same color scale)
    ax = axes[row_idx, 1]
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    sc = scatter_metric(ax, df_p, 'overall_rmse_norm', cmap='plasma_r',
                        vmin=RMSE_VMIN, vmax=RMSE_VMAX, size=60)
    plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label='RMSE (norm.)')
    ax.set_title(f'Persistence — {mr_tag}', fontsize=11)

    # Right: skill score
    ax = axes[row_idx, 2]
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    sc = scatter_metric(ax, df_sk, 'skill', cmap='RdBu',
                        vmin=-0.3, vmax=0.3, size=60)
    plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label='Skill score')
    ax.set_title(f'Skill score — {mr_tag}', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'map_persistence_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# ── Quick summary table ────────────────────────────────────────────────────
print('\nMedian per-station skill score (1 - model/persistence):')
print(f'  mr0.00  {df_skill_mr0["skill"].median():+.3f}   '
      f'({(df_skill_mr0["skill"] > 0).sum()}/{len(df_skill_mr0)} stations beat persistence)')
print(f'  mr0.50  {df_skill_mr5["skill"].median():+.3f}   '
      f'({(df_skill_mr5["skill"] > 0).sum()}/{len(df_skill_mr5)} stations beat persistence)')